# Detecting and Stress-Testing Action-Dependent Sensor Shortcuts in Deep Reinforcement Learning for Smart Irrigation

**Reproduction notebook.** 10-min decision steps and soil-MDP diagnostics, the action-dependent Markov chain of DHT22 health with a common-cause check, an audit of the replay simulator, and nine double-DQN variants (15 seeds each) scored under four sensor regimes with a counterfactual shortcut index, plus a robustness check in a band-constrained simulator (Tables 2-8, Figs. 1-6). All learners are written in numpy, so runs are identical across machines.

**Run time:** about 80 min for the whole notebook on a laptop, most of it in Steps 6 and 7; expect roughly twice as long on a free Colab CPU. Set `RUN_ROBUSTNESS = False` in Step 7 to skip the robustness check (its checks then show ❌).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badrelkhalylyphd-boop/irrigation-drl-fault-shortcuts/blob/main/reproduce_colab.ipynb)

### How to use
1. **Runtime > Run all.** The first cell installs the exact library versions used for the paper. On Colab the session then restarts by itself.
2. Upload `greenhouse_dataset.csv` in the **Files** panel of Colab (folder icon on the left; it goes to `/content`). The file is in `data/` of this repository or on [Mendeley Data](https://doi.org/10.17632/2tvpsht4s4.1). It survives the automatic restart of step 1.
3. **Runtime > Run all** a second time. Cell 2 reads `/content/greenhouse_dataset.csv` directly and checks its SHA-256.
4. The last section compares every number of the revised paper with this run and marks each one ✅ or ❌.

Locally (Jupyter), open the notebook from the repository root: the data file is picked up from `data/`.

## 1. Environment

In [ ]:
# Install the exact library versions used for the paper (runs once).
import os, sys, subprocess, importlib.metadata as md
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
PINNED = {'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'scipy': '1.15.3', 'matplotlib': '3.11.2', 'pyarrow': '20.0.0'}

def _version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

wrong = {p: _version(p) for p, v in PINNED.items() if _version(p) != v}
if wrong:
    print("Installing pinned versions (currently:", wrong, ") ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + [f"{p}=={v}" for p, v in PINNED.items()])
    if IN_COLAB:
        print("\nDone. The session restarts now. Then choose Runtime > Run all again.")
        os.kill(os.getpid(), 9)
    raise SystemExit("Pinned versions installed. Restart the kernel, then run all cells again.")
print("Environment OK -", sys.version.split()[0], PINNED)


## 2. Data

In [ ]:
# Get the dataset.
# On Colab: upload greenhouse_dataset.csv in the Files panel (left sidebar); it lands in /content and is read from there.
# Locally: the copy in data/ of the repository is used.
import hashlib, shutil, urllib.request
from pathlib import Path

COLAB_FILE = Path("/content/greenhouse_dataset.csv")   # file uploaded through the Colab Files panel
DATA_URL = ""  # optional, e.g. "https://raw.githubusercontent.com/badrelkhalylyphd-boop/irrigation-drl-fault-shortcuts/main/data/greenhouse_dataset.csv"
EXPECTED_SHA256 = "e64a904a981d323f67210138cfc078e98d1911e6d2bb42195bf76aa818c51aff"

WORK = Path("work").resolve()
SRC, OUT = WORK / "src", WORK / "outputs"
SRC.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
DATA = WORK / "greenhouse_dataset.csv"

candidates = [COLAB_FILE, *sorted(Path("/content").glob("greenhouse_dataset*.csv")),
              Path("data/greenhouse_dataset.csv"), Path("../data/greenhouse_dataset.csv"), Path("greenhouse_dataset.csv")]
found = [p for p in candidates if p.is_file()]
if found:
    shutil.copy(found[0], DATA)
    print("Dataset read from", found[0])
elif DATA.exists():
    print("Dataset already in", DATA)
elif DATA_URL:
    urllib.request.urlretrieve(DATA_URL, DATA)
elif IN_COLAB:
    from google.colab import files
    print("greenhouse_dataset.csv not found in /content. Select it now (or upload it in the Files panel and rerun this cell).")
    name = next(iter(files.upload()))
    Path(name).replace(DATA)
else:
    raise FileNotFoundError("Put greenhouse_dataset.csv in data/ or next to this notebook, or set DATA_URL.")

sha = hashlib.sha256(DATA.read_bytes()).hexdigest()
print("SHA-256:", sha)
print("Same file as in the paper." if sha == EXPECTED_SHA256 else "WARNING: this is not the file used in the paper.")


## 3. Code
The cells below write the scripts of `src/` unchanged into `work/src/`.

In [ ]:
%%writefile work/src/common_rl.py
"""Shared pieces: decision steps, tabular MDP, k-NN simulator, a small numpy MLP, policies.

The state of a 10-min step is (m, f): the median soil-moisture reading of the step and the share of
its readings at or below 49 % (dry readings). diagnostics.py shows that the previous decision, the
previous step and the hour add nothing to the prediction of the next step given (m, f, a).
Everything is numpy/scikit-learn so that runs are identical on a laptop and on Colab.
"""
import numpy as np
from sklearn.neighbors import NearestNeighbors

STEP_MIN = 10
DRY = 49            # a reading <= 49 % is "dry"; the controller never logs a decision above it
GAMMA = 0.95        # 10-min steps: effective horizon of about 20 steps (3.3 h)
LAMBDAS = (0.0, 0.02, 0.05, 0.1, 0.2)
MAIN_LAMBDA = 0.05
M_EDGES = (60, 70, 80, 90)          # moisture bands: <60, 60-70, 70-80, 80-90, >=90
F_EDGES = (1e-9, 0.1)               # dry share: 0, (0, 0.1], > 0.1
NF = len(F_EDGES) + 1
NS = (len(M_EDGES) + 1) * NF        # 15 states
MIN_SUPPORT = 3                     # a = 1 is allowed in a state only if logged at least this often
ALPHA = 1.0                         # shrinkage of action-specific estimates toward the state average
DAY_STEPS = 24 * 60 // STEP_MIN     # 144 steps per simulated day
BANDS = ["<60", "60-70", "70-80", "80-90", ">=90"]
FLEVELS = ["f=0", "0<f<=0.1", "f>0.1"]


def state_index(m, f):
    f = np.asarray(f, float)
    return np.digitize(m, M_EDGES) * NF + np.digitize(f, F_EDGES, right=True)


def state_label(s):
    return f"{BANDS[s // NF]} | {FLEVELS[s % NF]}"


def day_folds(steps):
    """Two folds of days, alternating after sorting days by number of decision steps."""
    cnt = steps.groupby("day")["a"].sum().reset_index().sort_values(["a", "day"], ascending=[False, True])
    return {day: i % 2 for i, day in enumerate(cnt["day"])}


# --------------------------------------------------------------------------- tabular MDP
class TabularMDP:
    """Transition kernel and expected next dry share estimated from logged transitions.
    Action-specific estimates are shrunk toward the state average with weight ALPHA, so an
    action never logged in a state is assumed to change nothing there."""

    def __init__(self, trans, alpha=ALPHA, min_support=MIN_SUPPORT):
        s, a, s1, f1 = (trans[c].to_numpy() for c in ("s", "a", "s1", "f1"))
        N = np.zeros((NS, 2, NS)); Fs = np.zeros((NS, 2)); C = np.zeros((NS, 2))
        np.add.at(N, (s, a, s1), 1); np.add.at(Fs, (s, a), f1); np.add.at(C, (s, a), 1)
        pooled, pooledF = N.sum(1), Fs.sum(1)
        self.P = np.zeros((NS, 2, NS)); self.F = np.zeros((NS, 2))
        for k in range(NS):
            n = pooled[k].sum()
            base = pooled[k] / n if n else np.eye(NS)[k]
            bf = pooledF[k] / n if n else 0.0
            for act in (0, 1):
                self.P[k, act] = (N[k, act] + alpha * base) / (C[k, act] + alpha)
                self.F[k, act] = (Fs[k, act] + alpha * bf) / (C[k, act] + alpha)
        self.C = C
        self.support = C[:, 1] >= min_support
        self.mu = np.bincount(s, minlength=NS) / len(s)

    def evaluate(self, pi, lam, mu=None):
        """pi[s] = probability of irrigating in s. Returns the normalized discounted value J
        (per-step scale), the discounted dry share and the decisions per day."""
        mu = self.mu if mu is None else mu
        Pp = (1 - pi)[:, None] * self.P[:, 0] + pi[:, None] * self.P[:, 1]
        Fp = (1 - pi) * self.F[:, 0] + pi * self.F[:, 1]
        occ = (1 - GAMMA) * mu @ np.linalg.inv(np.eye(NS) - GAMMA * Pp)
        dry, rate = occ @ Fp, occ @ pi
        return {"J": -(dry + lam * rate), "dry_share": dry, "decisions_per_day": rate * DAY_STEPS}

    def solve(self, lam, max_iter=100):
        """Optimal policy restricted to supported actions, by policy iteration (exact evaluation
        at each step; it reaches the same fixed point as value iteration)."""
        pi = np.zeros(NS)
        for _ in range(max_iter):
            Pp = (1 - pi)[:, None] * self.P[:, 0] + pi[:, None] * self.P[:, 1]
            Rp = -((1 - pi) * self.F[:, 0] + pi * (self.F[:, 1] + lam))
            V = np.linalg.solve(np.eye(NS) - GAMMA * Pp, Rp)
            Q0 = -self.F[:, 0] + GAMMA * self.P[:, 0] @ V
            Q1 = np.where(self.support, -self.F[:, 1] - lam + GAMMA * self.P[:, 1] @ V, -np.inf)
            new = (Q1 > Q0 + 1e-12).astype(float)
            if np.array_equal(new, pi):
                break
            pi = new
        return pi


# --------------------------------------------------------------------------- features, simulator
def features(m, f):
    return np.column_stack([(np.asarray(m, float) - 75.0) / 15.0, np.asarray(f, float) * 4.0])


class KNNSim:
    """Next-step model that replays logged transitions: given (state, action) it draws one of the
    k nearest logged transitions with that action and applies its moisture change and next dry share."""

    def __init__(self, trans, k=10):
        self.start = trans[["m", "f"]].to_numpy(float)
        self.nn, self.dm, self.f1 = {}, {}, {}
        for act in (0, 1):
            t = trans[trans["a"] == act]
            self.nn[act] = NearestNeighbors(n_neighbors=min(k, len(t))).fit(features(t["m"], t["f"]))
            self.dm[act] = (t["m1"] - t["m"]).to_numpy()
            self.f1[act] = t["f1"].to_numpy()

    def step(self, st, a, rng):
        new = st.copy()
        for act in (0, 1):
            idx = np.where(a == act)[0]
            if len(idx) == 0:
                continue
            nb = self.nn[act].kneighbors(features(st[idx, 0], st[idx, 1]), return_distance=False)
            pick = nb[np.arange(len(idx)), rng.integers(0, nb.shape[1], len(idx))]
            new[idx, 0] = np.clip(st[idx, 0] + self.dm[act][pick], 0, 100)
            new[idx, 1] = self.f1[act][pick]
        return new

    def reset(self, n, rng):
        return self.start[rng.integers(0, len(self.start), n)].copy()

    def run(self, policy, lam, episodes, rng, steps=DAY_STEPS):
        """Simulate `episodes` days in parallel; return per-step averages."""
        st = self.reset(episodes, rng)
        tot_r = tot_dry = tot_a = 0.0
        for _ in range(steps):
            a = policy(st, rng).astype(int)
            a[st[:, 1] <= 0] = 0                         # no logged decision without a dry reading
            st = self.step(st, a, rng)
            tot_dry += st[:, 1].sum(); tot_a += a.sum(); tot_r += (-st[:, 1] - lam * a).sum()
        n = episodes * steps
        return {"reward": tot_r / n, "dry_share": tot_dry / n, "decisions_per_day": tot_a / episodes}


# --------------------------------------------------------------------------- numpy MLP with Adam
class MLP:
    def __init__(self, sizes, rng, lr=5e-4):
        self.W = [rng.normal(0, np.sqrt(2 / i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros(o) for o in sizes[1:]]
        self.lr, self.t = lr, 0
        self.m = [np.zeros_like(p) for p in self.W + self.b]
        self.v = [np.zeros_like(p) for p in self.W + self.b]

    def forward(self, x):
        acts = [x]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            x = x @ W + b
            if i < len(self.W) - 1:
                x = np.maximum(x, 0)
            acts.append(x)
        return x, acts

    def predict(self, x):
        return self.forward(x)[0]

    def backward(self, acts, g):
        """Gradients of the parameters for an output gradient g."""
        gW, gb = [], []
        for i in range(len(self.W) - 1, -1, -1):
            gW.insert(0, acts[i].T @ g); gb.insert(0, g.sum(0))
            if i:
                g = (g @ self.W[i].T) * (acts[i] > 0)
        return gW + gb

    def adam(self, grads):
        self.t += 1
        for j, (p, gr) in enumerate(zip(self.W + self.b, grads)):
            self.m[j] = 0.9 * self.m[j] + 0.1 * gr
            self.v[j] = 0.999 * self.v[j] + 0.001 * gr ** 2
            p -= self.lr * (self.m[j] / (1 - 0.9 ** self.t)) / (np.sqrt(self.v[j] / (1 - 0.999 ** self.t)) + 1e-8)

    def update(self, x, a, target, pair=None, beta=0.0):
        """One Adam step on the Huber loss of Q(x, a) against target. With pair = (x_clean, x_fault)
        and beta > 0, adds beta * mean over the batch of sum_a (Q(x_clean, a) - Q(x_fault, a))^2."""
        out, acts = self.forward(x)
        err = out[np.arange(len(a)), a] - target
        g = np.zeros_like(out)
        g[np.arange(len(a)), a] = np.clip(err, -1, 1) / len(a)
        grads = self.backward(acts, g)
        if pair is not None and beta > 0:
            qc, ac = self.forward(pair[0]); qf, af = self.forward(pair[1])
            dq = 2 * beta * (qc - qf) / len(qc)
            gc, gf = self.backward(ac, dq), self.backward(af, -dq)
            grads = [x0 + x1 + x2 for x0, x1, x2 in zip(grads, gc, gf)]
        self.adam(grads)
        return float(np.mean(err ** 2))

    def copy_from(self, other):
        self.W = [w.copy() for w in other.W]; self.b = [b.copy() for b in other.b]


def q_masked(net, st):
    q = net.predict(features(st[:, 0], st[:, 1]))
    q[:, 1] = np.where(st[:, 1] > 0, q[:, 1], -np.inf)
    return q


# --------------------------------------------------------------------------- POMDP with action-dependent sensor faults
HUM_FAIL, TAIR_FAIL = 0.0, 0.2      # what the logger writes when the DHT22 read fails


class FaultPOMDP:
    """Hidden state: soil moisture m, dry share f, air humidity and temperature, DHT22 health H
    (0 healthy, 1 degraded, 2 failed). Soil dynamics replay logged transitions (k nearest
    neighbours on (m, f) with the same action); the air climate follows the chosen neighbour; H
    follows a Markov chain whose matrix depends on the action, P[a, H, H'].
    The agent sees (m, f, humidity, air temperature) where the two air readings carry the
    failure code when H = 2, and with probability 1/2 when H = 1."""

    def __init__(self, trans, P_fault, k=10, start_healthy=False, band_constrained=False, null_effect=False):
        """band_constrained: neighbours are searched within the moisture band of the state; where a
        band holds fewer than MIN_SUPPORT logged decisions, a decision behaves like no decision.
        null_effect: a decision always behaves like no decision (a simulator with no action effect)."""
        self.P = np.asarray(P_fault, float)
        self.cum = self.P.cumsum(2)
        self.start = trans[["m", "f", "hum_fill", "tair_fill", "H"]].to_numpy(float)
        if start_healthy:
            self.start[:, 4] = 0
        self.band_constrained = band_constrained
        bands = np.digitize(trans["m"], M_EDGES)
        self.models = {}
        for b in (range(len(M_EDGES) + 1) if band_constrained else [-1]):
            for act in (0, 1):
                sel = (trans["a"] == act) & ((bands == b) if b >= 0 else True)
                if act == 1 and (null_effect or sel.sum() < MIN_SUPPORT):
                    sel = (trans["a"] == 0) & ((bands == b) if b >= 0 else True)
                t = trans[sel]
                self.models[(b, act)] = (NearestNeighbors(n_neighbors=min(k, len(t))).fit(features(t["m"], t["f"])),
                                         (t["m1"] - t["m"]).to_numpy(), t["f1"].to_numpy(), t["hum1"].to_numpy(), t["tair1"].to_numpy())

    def reset(self, n, rng):
        return self.start[rng.integers(0, len(self.start), n)].copy()

    def neighbours(self, st, a):
        """Indices of the k nearest logged transitions for each (state, action), with the model used."""
        out = []
        band = np.digitize(st[:, 0], M_EDGES) if self.band_constrained else np.full(len(st), -1)
        for (b, act), model in self.models.items():
            idx = np.where((a == act) & (band == b))[0]
            if len(idx):
                out.append((idx, model, model[0].kneighbors(features(st[idx, 0], st[idx, 1]), return_distance=False)))
        return out

    def step(self, st, a, rng):
        new = st.copy()
        for idx, (nn, dm, f1, h1, t1), nb in self.neighbours(st, a):
            pick = nb[np.arange(len(idx)), rng.integers(0, nb.shape[1], len(idx))]
            new[idx, 0] = np.clip(st[idx, 0] + dm[pick], 0, 100)
            new[idx, 1] = f1[pick]; new[idx, 2] = h1[pick]; new[idx, 3] = t1[pick]
        u = rng.random(len(st))
        cum = self.cum[a, st[:, 4].astype(int)]
        new[:, 4] = (u[:, None] > cum).sum(1).clip(0, 2)
        return new

    @staticmethod
    def observe(st, rng):
        """Observed (m, f, humidity, air temperature) and the boolean 'failure code shown'."""
        fault = (st[:, 4] == 2) | ((st[:, 4] == 1) & (rng.random(len(st)) < 0.5))
        obs = st[:, :4].copy()
        obs[fault, 2] = HUM_FAIL; obs[fault, 3] = TAIR_FAIL
        return obs, fault


def obs_features(obs):
    """Scaled network input: (m, f, humidity, air temperature)."""
    return np.column_stack([(obs[:, 0] - 75.0) / 15.0, obs[:, 1] * 4.0, (obs[:, 2] - 80.0) / 20.0, (obs[:, 3] - 28.0) / 5.0])


def faulted(obs):
    out = obs.copy(); out[:, 2] = HUM_FAIL; out[:, 3] = TAIR_FAIL
    return out


# --------------------------------------------------------------------------- policies on simulator states
def p_never(st, rng):
    return np.zeros(len(st), int)


def p_any_dry(st, rng):
    return (st[:, 1] > 0).astype(int)


def p_tabular(pi):
    return lambda st, rng: pi[state_index(st[:, 0], st[:, 1])].round().astype(int)


def p_behaviour(clf):
    """Stochastic clone of the logged controller: logistic model of P(a = 1 | m, f) on dry steps."""
    def f(st, rng):
        p = np.zeros(len(st)); dry = st[:, 1] > 0
        if dry.any():
            p[dry] = clf.predict_proba(features(st[dry, 0], st[dry, 1]))[:, 1]
        return (rng.random(len(st)) < p).astype(int)
    return f


def p_net(net):
    return lambda st, rng: q_masked(net, st).argmax(1)


In [ ]:
%%writefile work/src/shortcut_core.py
"""Shared code of the DQN experiments: sensor regimes, agents, training, evaluation, shortcut index,
and seed-level statistics (mean, interquartile mean, bootstrap intervals)."""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from common_rl import (FaultPOMDP, MLP, TabularMDP, GAMMA, MAIN_LAMBDA, DAY_STEPS, features, obs_features,
                       faulted, state_index)

LAM = MAIN_LAMBDA
SEEDS = tuple(range(15))
N_ENV, ITERS, BATCH, SYNC, WARMUP = 16, 3000, 64, 100, 200
EVAL_EPISODES, EVAL_SEED, CSI_EPISODES = 400, 7, 100
DEFICIT_LEVEL = 60.0           # secondary outcome: mean shortfall of the step median below 60 %
NAMES = ["healthy", "degraded", "failed"]
REGIME_NAMES = ["source", "repaired", "action-independent", "coupled"]


def fault_matrices(path="fault_chain.csv"):
    fc = pd.read_csv(path)
    P = np.zeros((2, 3, 3)); N = np.zeros((2, 3, 3))
    for r in fc.itertuples():
        P[r.action, NAMES.index(r._2), NAMES.index(r.to)] = r.p
        N[r.action, NAMES.index(r._2), NAMES.index(r.to)] = r.count
    pooled = (N.sum(0) + 0.5) / (N.sum(0) + 0.5).sum(1, keepdims=True)
    P_ind = np.stack([pooled, pooled])
    P_rep = np.zeros((2, 3, 3)); P_rep[:, :, 0] = 1.0
    P_cpl = P.copy(); P_cpl[1, 0] = [0.4, 0.1, 0.5]
    return {"source": P, "repaired": P_rep, "action-independent": P_ind, "coupled": P_cpl}


def make_regimes(t, **sim_kw):
    Ps = fault_matrices()
    return {name: FaultPOMDP(t, Ps[name], start_healthy=(name == "repaired"), **sim_kw) for name in REGIME_NAMES}


class Agent:
    """Turns observations into network inputs; keeps the last valid air readings for 'ffill'."""

    def __init__(self, mode, net=None):
        self.mode, self.net, self.last = mode, net, None

    def reset(self, obs, st):
        self.last = st[:, 2:4].copy()

    def inputs(self, obs, fault):
        if self.mode == "mask":
            return features(obs[:, 0], obs[:, 1])
        if self.mode == "ffill":
            o = obs.copy()
            self.last[~fault] = obs[~fault, 2:4]
            o[fault, 2:4] = self.last[fault]
            return obs_features(o)
        return obs_features(obs)

    def counterfactual(self, st):
        if self.mode == "mask":
            x = features(st[:, 0], st[:, 1]); return x, x
        return obs_features(st[:, :4]), obs_features(faulted(st[:, :4]))

    def greedy(self, x, f):
        q = self.net.predict(x); q[:, 1] = np.where(f > 0, q[:, 1], -np.inf)
        return q.argmax(1)


def train(cfg, seed, regimes, curve=None, tag=None):
    """Double DQN; cfg: mode, beta (consistency weight), rho (reliability-aware exploration), train regime."""
    rng = np.random.default_rng(seed)
    env = regimes[cfg["train"]]
    din = 2 if cfg["mode"] == "mask" else 4
    net, tgt = MLP([din, 64, 64, 2], rng), MLP([din, 64, 64, 2], rng); tgt.copy_from(net)
    ag = Agent(cfg["mode"], net)
    cap = N_ENV * ITERS
    X = np.zeros((cap, din)); X1 = np.zeros((cap, din)); XC = np.zeros((cap, din)); XF = np.zeros((cap, din))
    A = np.zeros(cap, int); R = np.zeros(cap); F1 = np.zeros(cap); n = 0
    for it in range(ITERS):
        if it % DAY_STEPS == 0:
            st = env.reset(N_ENV, rng); obs, fault = env.observe(st, rng); ag.reset(obs, st)
            x = ag.inputs(obs, fault)
        eps = max(0.05, 1 - it / (0.5 * ITERS)) * (1 - cfg["rho"] * fault)
        a = ag.greedy(x, obs[:, 1])
        explore = rng.random(N_ENV) < eps
        a[explore] = rng.integers(0, 2, explore.sum())
        a[obs[:, 1] <= 0] = 0
        xc, xf = ag.counterfactual(st)
        st1 = env.step(st, a, rng); obs1, fault1 = env.observe(st1, rng); x1 = ag.inputs(obs1, fault1)
        sl = slice(n, n + N_ENV)
        X[sl], A[sl], R[sl], X1[sl], F1[sl], XC[sl], XF[sl] = x, a, -st1[:, 1] - LAM * a, x1, st1[:, 1], xc, xf
        n += N_ENV; st, obs, fault, x = st1, obs1, fault1, x1
        if n >= WARMUP:
            i = rng.integers(0, n, BATCH)
            q_on = net.predict(X1[i]); q_on[:, 1] = np.where(F1[i] > 0, q_on[:, 1], -np.inf)
            q_t = tgt.predict(X1[i])[np.arange(BATCH), q_on.argmax(1)]
            pair = (XC[i], XF[i]) if cfg["beta"] > 0 else None
            net.update(X[i], A[i], R[i] + GAMMA * q_t, pair=pair, beta=cfg["beta"])
        if it % SYNC == 0:
            tgt.copy_from(net)
        if curve is not None and (it + 1) % 500 == 0:
            curve.append({"config": tag, "seed": seed, "iteration": it + 1, **evaluate(ag, regimes["source"], 100)})
    return ag


def evaluate(policy, env, episodes=EVAL_EPISODES):
    """policy: an Agent, or a function (obs, rng) -> actions that ignores the air channels."""
    rng = np.random.default_rng(EVAL_SEED)
    st = env.reset(episodes, rng); obs, fault = env.observe(st, rng)
    if isinstance(policy, Agent):
        policy.reset(obs, st)
    tot = dict.fromkeys(["r", "dry", "dec", "deficit", "dec_fault", "dry_fault", "dec_clean", "dry_clean"], 0.0)
    for _ in range(DAY_STEPS):
        if isinstance(policy, Agent):
            a = policy.greedy(policy.inputs(obs, fault), obs[:, 1])
        else:
            a = policy(obs, rng).astype(int)
        a[obs[:, 1] <= 0] = 0
        dry = obs[:, 1] > 0
        tot["dec_fault"] += (a * (dry & fault)).sum(); tot["dry_fault"] += (dry & fault).sum()
        tot["dec_clean"] += (a * (dry & ~fault)).sum(); tot["dry_clean"] += (dry & ~fault).sum()
        st = env.step(st, a, rng); obs, fault = env.observe(st, rng)
        tot["r"] += (-st[:, 1] - LAM * a).sum(); tot["dry"] += st[:, 1].sum(); tot["dec"] += a.sum()
        tot["deficit"] += np.maximum(0, DEFICIT_LEVEL - st[:, 0]).sum()
    n = episodes * DAY_STEPS
    return {"reward": tot["r"] / n, "dry_share": tot["dry"] / n, "decisions_per_day": tot["dec"] / episodes,
            "deficit": tot["deficit"] / n,
            "p_decide_dry_fault": tot["dec_fault"] / max(tot["dry_fault"], 1),
            "p_decide_dry_clean": tot["dec_clean"] / max(tot["dry_clean"], 1)}


def csi(ag, env):
    """Counterfactual shortcut index along the agent's own greedy trajectories in env."""
    rng = np.random.default_rng(EVAL_SEED + 1)
    st = env.reset(CSI_EPISODES, rng); obs, fault = env.observe(st, rng); ag.reset(obs, st)
    flips = dq = n_dry = n_all = 0.0
    for _ in range(DAY_STEPS):
        xc, xf = ag.counterfactual(st)
        qc, qf = ag.net.predict(xc), ag.net.predict(xf)
        dq += np.abs(qc - qf).sum(); n_all += 2 * len(st)
        dry = st[:, 1] > 0
        flips += (qc[dry].argmax(1) != qf[dry].argmax(1)).sum(); n_dry += dry.sum()
        a = ag.greedy(ag.inputs(obs, fault), obs[:, 1]); a[obs[:, 1] <= 0] = 0
        st = env.step(st, a, rng); obs, fault = env.observe(st, rng)
    return {"flip_rate": flips / max(n_dry, 1), "mean_abs_dq": dq / n_all}


def baselines(t):
    dry_t = t[t["f"] > 0]
    clone = LogisticRegression().fit(features(dry_t["m"], dry_t["f"]), dry_t["a"])
    pi_vi = TabularMDP(t).solve(LAM)
    return {
        "never": lambda o, rng: np.zeros(len(o), int),
        "any dry reading": lambda o, rng: (o[:, 1] > 0).astype(int),
        "logged controller (clone)": lambda o, rng: (rng.random(len(o)) < np.where(
            o[:, 1] > 0, clone.predict_proba(features(o[:, 0], o[:, 1]))[:, 1], 0)).astype(int),
        "tabular plan": lambda o, rng: pi_vi[state_index(o[:, 0], o[:, 1])].round().astype(int),
    }


def run_experiment(t, regimes, configs, prefix, curve_configs=(), seeds=SEEDS):
    """Train every config with every seed, score in every regime, compute the shortcut index in the
    source regime, and write <prefix>_eval.csv, <prefix>_csi.csv, <prefix>_curve.csv, <prefix>_stats.csv."""
    rows, csis, curve = [], [], []
    for name, pol in baselines(t).items():
        for reg, env in regimes.items():
            rows.append({"config": name, "seed": -1, "regime": reg, **evaluate(pol, env)})
    for name, cfg in configs.items():
        for seed in seeds:
            ag = train(cfg, seed, regimes, curve if name in curve_configs else None, name)
            for reg, env in regimes.items():
                rows.append({"config": name, "seed": seed, "regime": reg, **evaluate(ag, env)})
            csis.append({"config": name, "seed": seed, **csi(ag, regimes["source"])})
        print(prefix, name, "done", flush=True)
    ev = pd.DataFrame(rows); ev.to_csv(f"{prefix}_eval.csv", index=False)
    cs = pd.DataFrame(csis); cs.to_csv(f"{prefix}_csi.csv", index=False)
    pd.DataFrame(curve).to_csv(f"{prefix}_curve.csv", index=False)

    # Seed-level quantities: reward per regime, worst regime, spread, shortcut index.
    piv = ev[ev.seed >= 0].pivot_table(index=["config", "seed"], columns="regime", values="reward")
    piv["worst"] = piv[REGIME_NAMES].min(1); piv["spread"] = piv[REGIME_NAMES].max(1) - piv[REGIME_NAMES].min(1)
    per_seed = piv.reset_index().merge(cs, on=["config", "seed"])
    per_seed.to_csv(f"{prefix}_per_seed.csv", index=False)
    rng = np.random.default_rng(2026)
    stats = []
    for name in configs:
        z = per_seed[per_seed.config == name]
        for metric in REGIME_NAMES + ["worst", "spread", "mean_abs_dq", "flip_rate"]:
            stats.append({"config": name, "metric": metric, **seed_stats(z[metric], rng)})
    pd.DataFrame(stats).to_csv(f"{prefix}_stats.csv", index=False)
    return ev, cs, per_seed


def iqm(x):
    x = np.sort(np.asarray(x, float)); n = len(x); lo, hi = int(np.floor(0.25 * n)), int(np.ceil(0.75 * n))
    return x[lo:hi].mean()


def seed_stats(values, rng, nboot=10000):
    """Mean, IQM and 95% percentile-bootstrap intervals over seeds."""
    v = np.asarray(values, float)
    idx = rng.integers(0, len(v), (nboot, len(v)))
    bm = v[idx].mean(1); bi = np.array([iqm(r) for r in v[idx]])
    return {"mean": v.mean(), "sd": v.std(ddof=1), "mean_lo": np.quantile(bm, .025), "mean_hi": np.quantile(bm, .975),
            "iqm": iqm(v), "iqm_lo": np.quantile(bi, .025), "iqm_hi": np.quantile(bi, .975), "n": len(v)}


In [ ]:
%%writefile work/src/rl_prep.py
"""Cleaning and 10-min decision steps.
Usage: python rl_prep.py greenhouse_dataset.csv
Outputs: steps.parquet (one row per 10-min step with at least 3 readings), transitions.parquet
(steps whose next step exists), prep_summary.csv

Cleaning is the one of the companion leakage-aware study: labeled rows with a readable timestamp
and five numeric sensor values (53,189 rows, 208 decisions). Duplicates are kept: on a time grid
they are repeated measurements.
"""
import sys
from datetime import datetime
import numpy as np
import pandas as pd
from common_rl import STEP_MIN, DRY, state_index, day_folds

FEATURES = ["Soil Percent", "Soil Raw", "Suhu Tanah", "Suhu Udara", "Humidity"]
MIN_READINGS = 3


def parse_ts(s):
    for fmt in ("%Y-%m-%d_%H-%M-%S", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(str(s), fmt)
        except ValueError:
            pass
    return pd.NaT


raw = pd.read_csv(sys.argv[1] if len(sys.argv) > 1 else "greenhouse_dataset.csv", sep=";")
d = raw[raw["Label"].notna()].copy()
d[FEATURES] = d[FEATURES].apply(pd.to_numeric, errors="coerce")
d["ts"] = pd.to_datetime(d["Timestamp"].map(parse_ts))
d = d.dropna(subset=FEATURES + ["ts"]).sort_values("ts", kind="mergesort")
d["Label"] = d["Label"].astype(int)
d["dry"] = d["Soil Percent"] <= DRY
d["dht_fail"] = (d["Humidity"] == 0) & (d["Suhu Udara"].round(1) == 0.2)   # DHT22 failure code
d["hum_ok"] = d["Humidity"].where(~d["dht_fail"]); d["tair_ok"] = d["Suhu Udara"].where(~d["dht_fail"])
assert d.loc[d["Label"] == 1, "Soil Percent"].max() <= DRY

d["bin"] = d["ts"].dt.floor(f"{STEP_MIN}min")
g = d.groupby("bin").agg(n=("Label", "size"), m=("Soil Percent", "median"), f=("dry", "mean"),
                         lo=("Soil Percent", "min"), a=("Label", "max"), n_dec=("Label", "sum"),
                         tsoil=("Suhu Tanah", "median"), dht_fail=("dht_fail", "mean"),
                         hum=("hum_ok", "median"), tair=("tair_ok", "median"))
g = g[g["n"] >= MIN_READINGS].copy()
# DHT22 health of the step: 0 healthy (no failure code), 1 degraded (<= half), 2 failed (> half).
g["H"] = np.select([g["dht_fail"] == 0, g["dht_fail"] <= 0.5], [0, 1], 2)
g["day"] = g.index.strftime("%Y-%m-%d")
# Climate of the step as the air sensor would have read it when working; failed steps take the
# day median of valid readings (used only as the hidden climate of the simulator).
for c in ("hum", "tair"):
    g[f"{c}_fill"] = g[c].fillna(g.groupby("day")[c].transform("median")).fillna(g[c].median())
step = pd.Timedelta(f"{STEP_MIN}min")
prev = g.reindex(g.index - step)
g["ap"] = prev["a"].fillna(0).astype(int).to_numpy()          # decision in the previous step
g["prev_exists"] = prev["m"].notna().to_numpy()
g["m_prev"] = prev["m"].to_numpy(); g["f_prev"] = prev["f"].to_numpy()
for k in (1, 2, 3):
    nx = g.reindex(g.index + k * step)
    g[f"m{k}"] = nx["m"].to_numpy(); g[f"f{k}"] = nx["f"].to_numpy(); g[f"H{k}"] = nx["H"].to_numpy()
nx = g.reindex(g.index + step)
g["hum1"] = nx["hum_fill"].to_numpy(); g["tair1"] = nx["tair_fill"].to_numpy()
g = g.reset_index().rename(columns={"bin": "t"})
g["hour"] = g["t"].dt.hour + g["t"].dt.minute / 60
g["s"] = state_index(g["m"], g["f"])
folds = day_folds(g)
g["fold"] = g["day"].map(folds)
g.to_parquet("steps.parquet", index=False)

t = g.dropna(subset=["m1"]).copy()
t["s1"] = state_index(t["m1"], t["f1"])
t.to_parquet("transitions.parquet", index=False)

summary = {
    "clean_readings": len(d), "decisions": int(d["Label"].sum()),
    "steps": len(g), "decision_steps": int(g["a"].sum()), "transitions": len(t),
    "decision_transitions": int(t["a"].sum()), "days": g["day"].nunique(),
    "steps_with_dry_reading": int((g["f"] > 0).sum()),
    "decision_steps_with_median_ge_50": int(((g["a"] == 1) & (g["m"] >= 50)).sum()),
    "steps_with_median_lt_50": int((g["m"] < 50).sum()),
    "mean_dry_share": float(g["f"].mean()),
    "steps_healthy": int((g["H"] == 0).sum()), "steps_degraded": int((g["H"] == 1).sum()), "steps_failed": int((g["H"] == 2).sum()),
    "decision_steps_failed_or_degraded": int(((g["a"] == 1) & (g["H"] > 0)).sum()),
    "fold0_days": int((pd.Series(folds) == 0).sum()), "fold1_days": int((pd.Series(folds) == 1).sum()),
    "fold0_decision_steps": int(g.loc[g["fold"] == 0, "a"].sum()), "fold1_decision_steps": int(g.loc[g["fold"] == 1, "a"].sum()),
    "fold0_transitions": int((t["fold"] == 0).sum()), "fold1_transitions": int((t["fold"] == 1).sum()),
}
pd.Series(summary).to_csv("prep_summary.csv", header=["value"])
print(pd.Series(summary).to_string())
print("folds:", dict(sorted(folds.items())))


In [ ]:
%%writefile work/src/diagnostics.py
"""Diagnostics that decide what an MDP built from the log can support.
Usage: python diagnostics.py
Requires steps.parquet and transitions.parquet (rl_prep.py).
Outputs: diag_behaviour.csv, diag_coverage.csv, diag_effect.csv, diag_markov.csv
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from common_rl import NS, BANDS, state_label, M_EDGES

SEED, NBOOT = 2026, 2000
g = pd.read_parquet("steps.parquet")
t = pd.read_parquet("transitions.parquet")
g["band"] = np.digitize(g["m"], M_EDGES); t["band"] = np.digitize(t["m"], M_EDGES)

# 1. Behaviour of the logged controller.
rows = []
for b, name in enumerate(BANDS):
    z = g[g["band"] == b]
    rows.append({"band": name, "steps": len(z), "dry_steps": int((z["f"] > 0).sum()), "decision_steps": int(z["a"].sum()),
                 "p_decision_given_dry": z.loc[z["f"] > 0, "a"].mean() if (z["f"] > 0).any() else np.nan,
                 "mean_dry_share_when_dry": z.loc[z["f"] > 0, "f"].mean()})
z = g[g["f"] > 0]
rows.append({"band": "all", "steps": len(g), "dry_steps": len(z), "decision_steps": int(g["a"].sum()),
             "p_decision_given_dry": z["a"].mean(), "mean_dry_share_when_dry": z["f"].mean()})
beh = pd.DataFrame(rows); beh.to_csv("diag_behaviour.csv", index=False)
print(beh.round(3).to_string(index=False))

# 2. Coverage of state-action pairs.
C = pd.crosstab(t["s"], t["a"]).reindex(range(NS), fill_value=0)
cov = pd.DataFrame({"state": range(NS), "label": [state_label(s) for s in range(NS)],
                    "n_a0": C.get(0, 0).to_numpy(), "n_a1": C.get(1, 0).to_numpy()})
cov.to_csv("diag_coverage.csv", index=False)

# 3. Effect of a logged decision, compared within state (20 tabular states).
t["d1"] = t["m1"] - t["m"]; t["d3"] = t["m3"] - t["m"]
OUTCOMES = {"d1": "moisture change, next step", "f1": "dry share, next step",
            "d3": "moisture change, 3 steps", "f3": "dry share, 3 steps ahead"}


def within_state(df, y):
    df = df.dropna(subset=[y])
    m = df.groupby(["s", "a"])[y].agg(["mean", "size"]).unstack()
    both = m.dropna()
    if both.empty:
        return np.nan, np.nan
    diff = both[("mean", 1)] - both[("mean", 0)]
    w = both[("size", 1)]
    naive = df.loc[df["a"] == 1, y].mean() - df.loc[df["a"] == 0, y].mean()
    return float((diff * w).sum() / w.sum()), float(naive)


rng = np.random.default_rng(SEED)
days = t["day"].unique()
by_day = {d: t[t["day"] == d] for d in days}
eff = []
for y, label in OUTCOMES.items():
    est, naive = within_state(t, y)
    boot = []
    for _ in range(NBOOT):
        sample = pd.concat([by_day[d] for d in rng.choice(days, len(days), replace=True)])
        boot.append(within_state(sample, y)[0])
    boot = np.array(boot); boot = boot[~np.isnan(boot)]
    n1 = int(t.dropna(subset=[y])["a"].sum())
    eff.append({"outcome": y, "label": label, "within_state": est, "naive": naive, "lo": np.quantile(boot, .025),
                "hi": np.quantile(boot, .975), "decision_transitions": n1, "replicates": len(boot)})
eff = pd.DataFrame(eff); eff.to_csv("diag_effect.csv", index=False)
print(eff.round(4).to_string(index=False))

# 4. Markov order: does the past add to (m, f, a) when predicting the next step?
u = t[t["prev_exists"]].dropna(subset=["m_prev", "f_prev"]).copy()
u["sin"] = np.sin(2 * np.pi * u["hour"] / 24); u["cos"] = np.cos(2 * np.pi * u["hour"] / 24)
SETS = {"M0: m, f, a": ["m", "f", "a"],
        "M1: + previous decision": ["m", "f", "a", "ap"],
        "M2: + previous m and f": ["m", "f", "a", "ap", "m_prev", "f_prev"],
        "M3: + hour of day": ["m", "f", "a", "ap", "m_prev", "f_prev", "sin", "cos"]}
mk = []
for target in ("d1", "f1"):
    for name, cols in SETS.items():
        for mname, make in (("linear", lambda: LinearRegression()),
                            ("boosting", lambda: GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=SEED))):
            pred = np.zeros(len(u))
            for d in u["day"].unique():
                tr, te = u["day"] != d, u["day"] == d
                pred[te.to_numpy()] = make().fit(u.loc[tr, cols], u.loc[tr, target]).predict(u.loc[te, cols])
            mk.append({"target": target, "features": name, "model": mname, "MAE": mean_absolute_error(u[target], pred), "n": len(u)})
# 5. Do valid air readings carry information on the soil once (m, f, a) is known?
v = t[t["H"] == 0].dropna(subset=["hum", "tair"]).copy()
for target in ("d1", "f1"):
    for name, cols in {"A0: m, f, a": ["m", "f", "a"], "A1: + humidity": ["m", "f", "a", "hum"],
                       "A2: + humidity, air temperature": ["m", "f", "a", "hum", "tair"]}.items():
        for mname, make in (("linear", lambda: LinearRegression()),
                            ("boosting", lambda: GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=SEED))):
            pred = np.zeros(len(v))
            for d in v["day"].unique():
                tr, te = v["day"] != d, v["day"] == d
                pred[te.to_numpy()] = make().fit(v.loc[tr, cols], v.loc[tr, target]).predict(v.loc[te, cols])
            mk.append({"target": target, "features": name, "model": mname, "MAE": mean_absolute_error(v[target], pred), "n": len(v)})
mk = pd.DataFrame(mk); mk.to_csv("diag_markov.csv", index=False)
print(mk.pivot_table(index=["target", "features"], columns="model", values="MAE").round(4).to_string())


In [ ]:
%%writefile work/src/fault_chain.py
"""Action-dependent Markov chain of DHT22 health, estimated from the log.
Usage: python fault_chain.py
Requires steps.parquet (rl_prep.py).
Outputs: fault_chain.csv (P^(a)(H' | H) with counts), fault_onset.csv (onset and recovery
probabilities by action with day-bootstrap intervals), fault_context.csv

Health of a 10-min step: H = 0 healthy (no failure code), 1 degraded (failure code in at most
half of the readings), 2 failed (more than half).
"""
import numpy as np
import pandas as pd

SEED, NBOOT, ALPHA = 2026, 2000, 0.5
NAMES = ["healthy", "degraded", "failed"]
g = pd.read_parquet("steps.parquet")
t = g.dropna(subset=["H1"]).copy(); t["H1"] = t["H1"].astype(int)


def matrices(df):
    N = np.zeros((2, 3, 3))
    np.add.at(N, (df["a"].to_numpy(int), df["H"].to_numpy(int), df["H1"].to_numpy(int)), 1)
    return N, (N + ALPHA) / (N + ALPHA).sum(2, keepdims=True)


N, P = matrices(t)
rows = []
for a in (0, 1):
    for i in range(3):
        for j in range(3):
            rows.append({"action": a, "from": NAMES[i], "to": NAMES[j], "count": int(N[a, i, j]), "p": P[a, i, j]})
pd.DataFrame(rows).to_csv("fault_chain.csv", index=False)


def quantities(df):
    """Onset within one step and within three steps (30 min) from a healthy step, and recovery
    from a failed step, by action."""
    out = {}
    for a in (0, 1):
        h = df[(df["H"] == 0) & (df["a"] == a)]
        out[f"onset1_a{a}"] = (h["H1"] > 0).mean() if len(h) else np.nan
        h3 = h.dropna(subset=["H2", "H3"])
        out[f"onset3_a{a}"] = ((h3[["H1", "H2", "H3"]] > 0).any(axis=1)).mean() if len(h3) else np.nan
        fl = df[(df["H"] == 2) & (df["a"] == a)]
        out[f"recover1_a{a}"] = (fl["H1"] == 0).mean() if len(fl) else np.nan
        out[f"n_healthy_a{a}"] = len(h); out[f"n_healthy3_a{a}"] = len(h3); out[f"n_failed_a{a}"] = len(fl)
    return out


est = quantities(t)
rng = np.random.default_rng(SEED)
days = t["day"].unique(); by_day = {d: t[t["day"] == d] for d in days}
boot = pd.DataFrame([quantities(pd.concat([by_day[d] for d in rng.choice(days, len(days), replace=True)]))
                     for _ in range(NBOOT)])
res = []
for q in ("onset1", "onset3", "recover1"):
    diff = boot[f"{q}_a1"] - boot[f"{q}_a0"]
    res.append({"quantity": q, "a0": est[f"{q}_a0"], "a1": est[f"{q}_a1"], "n_a0": est[f"n_{'failed' if q == 'recover1' else 'healthy3' if q == 'onset3' else 'healthy'}_a0"],
                "n_a1": est[f"n_{'failed' if q == 'recover1' else 'healthy3' if q == 'onset3' else 'healthy'}_a1"],
                "diff": est[f"{q}_a1"] - est[f"{q}_a0"], "diff_lo": diff.quantile(.025), "diff_hi": diff.quantile(.975),
                "p_diff_positive": (diff > 0).mean(), "replicates": int(diff.notna().sum())})
res = pd.DataFrame(res); res.to_csv("fault_onset.csv", index=False)

# Common-cause check: if an electrical glitch produced both a low soil reading (hence a decision)
# and a DHT22 failure, failures should also follow dry readings that carry no decision.
def onset_by_dry(df):
    h = df[(df["H"] == 0) & (df["a"] == 0)].dropna(subset=["H2", "H3"])
    on3 = (h[["H1", "H2", "H3"]] > 0).any(axis=1)
    on1 = h["H1"] > 0
    dry = h["f"] > 0
    return {"onset1_dry": on1[dry].mean(), "onset1_nodry": on1[~dry].mean(),
            "onset3_dry": on3[dry].mean(), "onset3_nodry": on3[~dry].mean(),
            "n_dry": int(dry.sum()), "n_nodry": int((~dry).sum())}


cc = onset_by_dry(t)
bcc = pd.DataFrame([onset_by_dry(pd.concat([by_day[d] for d in rng.choice(days, len(days), replace=True)]))
                    for _ in range(NBOOT)])
d3 = bcc["onset3_dry"] - bcc["onset3_nodry"]
cc.update({"diff3": cc["onset3_dry"] - cc["onset3_nodry"], "diff3_lo": d3.quantile(.025), "diff3_hi": d3.quantile(.975),
           "p_diff3_positive": (d3 > 0).mean()})
# Dry readings by health state: a failed air sensor coincides with low soil readings?
for h in range(3):
    z = g[g["H"] == h]
    cc[f"share_dry_reading_{NAMES[h]}"] = (z["f"] > 0).mean()
    cc[f"mean_dry_share_{NAMES[h]}"] = z["f"].mean()
pd.Series(cc).to_csv("fault_commoncause.csv", header=["value"])
print(pd.Series(cc).round(4).to_string())

ctx = pd.DataFrame([{"health": NAMES[h], "steps": int((g["H"] == h).sum()), "decision_steps": int(((g["H"] == h) & (g["a"] == 1)).sum()),
                     "dry_reading_steps": int(((g["H"] == h) & (g["f"] > 0)).sum())} for h in range(3)])
ctx.to_csv("fault_context.csv", index=False)
print(pd.DataFrame(rows).pivot_table(index=["action", "from"], columns="to", values="p").round(3))
print(res.round(4).to_string(index=False)); print(ctx.to_string(index=False))


In [ ]:
%%writefile work/src/mdp_planning.py
"""Planning on the tabular MDP estimated from the log, and three ways to evaluate the plan.
Usage: python mdp_planning.py
Requires transitions.parquet (rl_prep.py).
Outputs: mdp_policies.csv, mdp_insample.csv, mdp_bootstrap.csv, mdp_bootstrap_draws.csv,
         mdp_stability.csv, mdp_crossfit.csv

Reward per step: -(share of dry readings in the next step) - lambda * decision.
Policies: never irrigate; the logged controller (empirical P(a = 1 | s)); irrigate whenever a dry
reading appears; value iteration (VI) on the estimated MDP, restricted to supported actions.
"""
import numpy as np
import pandas as pd
from common_rl import TabularMDP, NS, LAMBDAS, MAIN_LAMBDA, state_label, NF

SEED, NBOOT = 2026, 500
t = pd.read_parquet("transitions.parquet")
full = TabularMDP(t)
pi_beh = np.where(full.C.sum(1) > 0, full.C[:, 1] / np.maximum(full.C.sum(1), 1), 0.0)
dry_state = (np.arange(NS) % NF) > 0
FIXED = {"never": np.zeros(NS), "logged controller": pi_beh, "irrigate on any dry reading": dry_state.astype(float)}

# Policies and in-sample values.
pol_rows, ins_rows, vi = [], [], {}
for lam in LAMBDAS:
    vi[lam] = full.solve(lam)
    for s in range(NS):
        pol_rows.append({"lambda": lam, "state": s, "label": state_label(s), "n_a0": int(full.C[s, 0]),
                         "n_a1": int(full.C[s, 1]), "supported": bool(full.support[s]), "p_logged": pi_beh[s], "vi": int(vi[lam][s])})
    for name, pi in {**FIXED, "value iteration": vi[lam]}.items():
        ins_rows.append({"lambda": lam, "policy": name, **full.evaluate(pi, lam)})
pd.DataFrame(pol_rows).to_csv("mdp_policies.csv", index=False)
ins = pd.DataFrame(ins_rows); ins.to_csv("mdp_insample.csv", index=False)
print(ins.round(4).to_string(index=False))

# Day bootstrap: rebuild the MDP from resampled days, evaluate the fixed full-data policies,
# and re-solve to see how stable the optimal policy is.
rng = np.random.default_rng(SEED)
days = t["day"].unique()
by_day = {d: t[t["day"] == d] for d in days}
draws, stab = [], np.zeros((len(LAMBDAS), NS))
for b in range(NBOOT):
    mb = TabularMDP(pd.concat([by_day[d] for d in rng.choice(days, len(days), replace=True)]))
    for i, lam in enumerate(LAMBDAS):
        row = {"rep": b, "lambda": lam}
        for name, pi in {**FIXED, "value iteration": vi[lam]}.items():
            row[name] = mb.evaluate(pi, lam)["J"]
        row["re-solved VI"] = mb.evaluate(mb.solve(lam), lam)["J"]
        stab[i] += mb.solve(lam)
        draws.append(row)
dr = pd.DataFrame(draws); dr.to_csv("mdp_bootstrap_draws.csv", index=False)
summ = []
for lam in LAMBDAS:
    z = dr[dr["lambda"] == lam]
    for base in ("never", "logged controller", "irrigate on any dry reading"):
        diff = z["value iteration"] - z[base]
        summ.append({"lambda": lam, "comparison": f"VI - {base}", "in_sample": float(ins[(ins["lambda"] == lam) & (ins.policy == "value iteration")].J.iloc[0]
                     - ins[(ins["lambda"] == lam) & (ins.policy == base)].J.iloc[0]),
                     "median": diff.median(), "lo": diff.quantile(.025), "hi": diff.quantile(.975), "p_positive": (diff > 0).mean()})
summ = pd.DataFrame(summ); summ.to_csv("mdp_bootstrap.csv", index=False)
print(summ.round(5).to_string(index=False))
st = pd.DataFrame(stab / NBOOT, columns=[state_label(s) for s in range(NS)]); st.insert(0, "lambda", LAMBDAS)
st.to_csv("mdp_stability.csv", index=False)

# Cross-fitting over two folds of days: plan on one fold, evaluate on the model of the other.
cf = []
for k in (0, 1):
    train, test = TabularMDP(t[t["fold"] == k]), TabularMDP(t[t["fold"] != k])
    for lam in LAMBDAS:
        pi = train.solve(lam)
        cf.append({"plan_fold": k, "lambda": lam, "policy": "value iteration", "irrigated_states": int(pi.sum()),
                   "J_plan_fold": train.evaluate(pi, lam)["J"], **{f"heldout_{key}": v for key, v in test.evaluate(pi, lam).items()}})
        for name, p in FIXED.items():
            cf.append({"plan_fold": k, "lambda": lam, "policy": name, "irrigated_states": int((p > 0).sum()),
                       "J_plan_fold": train.evaluate(p, lam)["J"], **{f"heldout_{key}": v for key, v in test.evaluate(p, lam).items()}})
cf = pd.DataFrame(cf); cf.to_csv("mdp_crossfit.csv", index=False)
print(cf.round(4).to_string(index=False))


In [ ]:
%%writefile work/src/sim_audit.py
"""Audit of the simulators: how large is the effect of a decision in each model, and why do the
fixed policies score on different scales in the tabular MDP and in the k-NN simulator?
Usage: python sim_audit.py
Requires transitions.parquet, fault_chain.csv.
Outputs: audit_effect.csv, audit_policies.csv

Models compared (soil part only; the air sensor plays no role here):
  replay k=10 (main), replay k=5, replay k=30, band-constrained k=10, null-effect (a decision
  replays transitions without a decision), and the tabular MDP.
"""
import numpy as np
import pandas as pd
from common_rl import TabularMDP, NS, NF, M_EDGES, BANDS, DAY_STEPS, GAMMA, MAIN_LAMBDA, features
from common_rl import FaultPOMDP
from shortcut_core import fault_matrices, baselines, evaluate, EVAL_EPISODES, EVAL_SEED

t = pd.read_parquet("transitions.parquet")
P_rep = fault_matrices()["repaired"]
SIMS = {"replay k=10": dict(k=10), "replay k=5": dict(k=5), "replay k=30": dict(k=30),
        "band-constrained k=10": dict(k=10, band_constrained=True), "null effect k=10": dict(k=10, null_effect=True)}
sims = {name: FaultPOMDP(t, P_rep, start_healthy=True, **kw) for name, kw in SIMS.items()}
tab = TabularMDP(t)

# 1. One-step effect of a decision on the expected next dry share, at every logged dry step.
dry = t[t["f"] > 0].copy()
st = np.column_stack([dry["m"], dry["f"], dry["hum_fill"], dry["tair_fill"], np.zeros(len(dry))])
band = np.digitize(dry["m"], M_EDGES)
rows = []


def expected_f1(sim, a):
    out = np.zeros(len(st))
    for idx, model, nb in sim.neighbours(st, np.full(len(st), a)):
        out[idx] = model[2][nb].mean(1)
    return out


effects = {name: expected_f1(sim, 1) - expected_f1(sim, 0) for name, sim in sims.items()}
effects["tabular MDP"] = tab.F[dry["s"].to_numpy(), 1] - tab.F[dry["s"].to_numpy(), 0]
# Where do the a = 1 neighbours of the main simulator come from?
nn1, _, _, _, _ = sims["replay k=10"].models[(-1, 1)]
nn0 = sims["replay k=10"].models[(-1, 0)][0]
d1, i1 = nn1.kneighbors(features(st[:, 0], st[:, 1]))
d0, _ = nn0.kneighbors(features(st[:, 0], st[:, 1]))
dec = t[t["a"] == 1]
other_band = (np.digitize(dec["m"].to_numpy()[i1], M_EDGES) != band[:, None]).mean(1)
wetter = (dec["m"].to_numpy()[i1] > dry["m"].to_numpy()[:, None] + 10).mean(1)
for b in list(range(len(BANDS))) + ["all"]:
    sel = np.ones(len(dry), bool) if b == "all" else band == b
    row = {"band": "all" if b == "all" else BANDS[b], "dry_steps": int(sel.sum())}
    for name, e in effects.items():
        row[f"effect: {name}"] = e[sel].mean()
    row["mean distance to a=1 neighbours"] = d1[sel].mean()
    row["mean distance to a=0 neighbours"] = d0[sel].mean()
    row["share of a=1 neighbours from another band"] = other_band[sel].mean()
    row["share of a=1 neighbours >10 points wetter"] = wetter[sel].mean()
    rows.append(row)
eff = pd.DataFrame(rows); eff.to_csv("audit_effect.csv", index=False)

# 2. Fixed policies in every model, on a common footing.
pols = baselines(t)
mu = tab.mu
pi_beh = np.where(tab.C.sum(1) > 0, tab.C[:, 1] / np.maximum(tab.C.sum(1), 1), 0.0)
TAB_PI = {"never": np.zeros(NS), "any dry reading": ((np.arange(NS) % NF) > 0).astype(float),
          "logged controller (clone)": pi_beh, "tabular plan": tab.solve(MAIN_LAMBDA)}


def tabular_days(pi, episodes=EVAL_EPISODES, seed=EVAL_SEED):
    """Undiscounted 144-step days in the tabular chain, from the empirical state distribution."""
    rng = np.random.default_rng(seed)
    s = rng.choice(NS, size=episodes, p=mu); dry_tot = dec_tot = 0.0
    cumP = tab.P.cumsum(2)
    for _ in range(DAY_STEPS):
        a = (rng.random(episodes) < pi[s]).astype(int)
        dry_tot += tab.F[s, a].sum(); dec_tot += a.sum()
        s = np.minimum((rng.random(episodes)[:, None] > cumP[s, a]).sum(1), NS - 1)
    n = episodes * DAY_STEPS
    return {"dry_share": dry_tot / n, "decisions_per_day": dec_tot / episodes, "reward": -(dry_tot + MAIN_LAMBDA * dec_tot) / n}


prow = []
for name, pi in TAB_PI.items():
    j = tab.evaluate(pi, MAIN_LAMBDA)
    prow.append({"policy": name, "model": "tabular MDP, discounted J", "reward": j["J"], "dry_share": j["dry_share"],
                 "decisions_per_day": j["decisions_per_day"]})
    prow.append({"policy": name, "model": "tabular MDP, 144-step days", **tabular_days(pi)})
for sname, sim in sims.items():
    for name, pol in pols.items():
        r = evaluate(pol, sim)
        prow.append({"policy": name, "model": sname, "reward": r["reward"], "dry_share": r["dry_share"],
                     "decisions_per_day": r["decisions_per_day"], "deficit": r["deficit"]})
pol_df = pd.DataFrame(prow); pol_df.to_csv("audit_policies.csv", index=False)
pd.set_option("display.width", 220)
print(eff.round(4).T.to_string())
print(pol_df.pivot_table(index="model", columns="policy", values="dry_share").round(4).to_string())
print(pol_df.pivot_table(index="model", columns="policy", values="decisions_per_day").round(2).to_string())


In [ ]:
%%writefile work/src/drl_shortcut.py
"""Shortcut detection and remedies for double DQN in a POMDP with action-dependent sensor faults.
Usage: python drl_shortcut.py
Requires transitions.parquet (rl_prep.py) and fault_chain.csv (fault_chain.py).
Outputs: shortcut_eval.csv, shortcut_csi.csv, shortcut_curve.csv, shortcut_per_seed.csv,
shortcut_stats.csv (mean, IQM, bootstrap intervals over 15 seeds), shortcut_tests.csv

Agents (double DQN, 15 seeds each, lambda = MAIN_LAMBDA), all trained in the source regime unless stated:
  full        sees (m, f, humidity, air temperature), failure codes included
  mask        sees (m, f) only
  ffill       air readings carrying the failure code are replaced by the last valid ones
  sc          SC-DQN: full input + consistency penalty (beta = 1) between the clean and the
              faulted version of each stored observation
  sc+RAE      SC-DQN with reliability-aware exploration (epsilon x 0.2 when a failure code is shown)
  full+RAE    plain DQN with reliability-aware exploration
  sc-noAD     SC-DQN trained with action-independent faults (pooled chain)
  sc-b0.1, sc-b10   SC-DQN with consistency weight 0.1 and 10
The simulator replays logged soil transitions (k nearest neighbours on (m, f), same action).
"""
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from shortcut_core import make_regimes, run_experiment

CONFIGS = {
    "full": dict(mode="full", beta=0.0, rho=0.0, train="source"),
    "mask": dict(mode="mask", beta=0.0, rho=0.0, train="source"),
    "ffill": dict(mode="ffill", beta=0.0, rho=0.0, train="source"),
    "sc": dict(mode="full", beta=1.0, rho=0.0, train="source"),
    "sc+RAE": dict(mode="full", beta=1.0, rho=0.8, train="source"),
    "full+RAE": dict(mode="full", beta=0.0, rho=0.8, train="source"),
    "sc-noAD": dict(mode="full", beta=1.0, rho=0.0, train="action-independent"),
    "sc-b0.1": dict(mode="full", beta=0.1, rho=0.0, train="source"),
    "sc-b10": dict(mode="full", beta=10.0, rho=0.0, train="source"),
}

t = pd.read_parquet("transitions.parquet")
ev, cs, per_seed = run_experiment(t, make_regimes(t), CONFIGS, "shortcut", curve_configs=("full", "sc"))

# One-sided Mann-Whitney tests between independent sets of seeds.
tests = []
for a, b, metric, alt in [("sc", "full", "spread", "less"), ("sc", "full", "mean_abs_dq", "less"),
                          ("sc", "full", "worst", "greater"), ("sc", "full", "coupled", "greater"),
                          ("sc", "full+RAE", "mean_abs_dq", "less"), ("full+RAE", "full", "flip_rate", "greater"),
                          ("ffill", "sc", "worst", "two-sided"), ("sc-b0.1", "sc", "source", "two-sided")]:
    x = per_seed.loc[per_seed.config == a, metric]; y = per_seed.loc[per_seed.config == b, metric]
    u, p = mannwhitneyu(x, y, alternative=alt)
    tests.append({"a": a, "b": b, "metric": metric, "alternative": alt, "U": u, "p": p,
                  "median_a": x.median(), "median_b": y.median()})
tests = pd.DataFrame(tests); tests.to_csv("shortcut_tests.csv", index=False)
pd.set_option("display.width", 220)
st = pd.read_csv("shortcut_stats.csv")
print(st.pivot_table(index="config", columns="metric", values="mean").round(4).to_string())
print(tests.round(4).to_string(index=False))


In [ ]:
%%writefile work/src/drl_robustness.py
"""Robustness check: the main agents retrained in a band-constrained simulator.
Usage: python drl_robustness.py
Requires transitions.parquet and fault_chain.csv.
Outputs: robust_eval.csv, robust_csi.csv, robust_per_seed.csv, robust_stats.csv, robust_tests.csv

In the band-constrained simulator a decision draws its outcome only from logged decisions in the
same moisture band; in bands with fewer than three logged decisions a decision has no effect.
This removes the borrowing of outcomes from decisions logged in wetter soil (see sim_audit.py).
The band-constrained simulator is about three times slower, so this check uses 10 seeds.
"""
import pandas as pd
from scipy.stats import mannwhitneyu
from shortcut_core import make_regimes, run_experiment

CONFIGS = {
    "full": dict(mode="full", beta=0.0, rho=0.0, train="source"),
    "mask": dict(mode="mask", beta=0.0, rho=0.0, train="source"),
    "ffill": dict(mode="ffill", beta=0.0, rho=0.0, train="source"),
    "sc": dict(mode="full", beta=1.0, rho=0.0, train="source"),
}
t = pd.read_parquet("transitions.parquet")
ev, cs, per_seed = run_experiment(t, make_regimes(t, band_constrained=True), CONFIGS, "robust", seeds=tuple(range(10)))
tests = []
for metric, alt in (("spread", "less"), ("mean_abs_dq", "less"), ("worst", "greater")):
    x = per_seed.loc[per_seed.config == "sc", metric]; y = per_seed.loc[per_seed.config == "full", metric]
    u, p = mannwhitneyu(x, y, alternative=alt)
    tests.append({"a": "sc", "b": "full", "metric": metric, "alternative": alt, "U": u, "p": p,
                  "median_a": x.median(), "median_b": y.median()})
pd.DataFrame(tests).to_csv("robust_tests.csv", index=False)
print(pd.read_csv("robust_stats.csv").pivot_table(index="config", columns="metric", values="mean").round(4).to_string())
print(pd.DataFrame(tests).round(4).to_string(index=False))


In [ ]:
%%writefile work/src/figures_rl.py
"""Figures of the paper.
Usage: python figures_rl.py  (after the other scripts)
Output: figures/fig1_behaviour.png ... figures/fig6_csi_curves.png
"""
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common_rl import BANDS, FLEVELS, NF, LAMBDAS

OUT = Path("figures"); OUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False})
LABEL = {"full": "DQN", "mask": "DQN + masking", "ffill": "DQN + last valid", "sc": "SC-DQN",
         "sc+RAE": "SC-DQN + RAE", "full+RAE": "DQN + RAE", "sc-noAD": "SC-DQN, action-indep. training",
         "sc-b0.1": "SC-DQN β=0.1", "sc-b10": "SC-DQN β=10"}
ORDER = list(LABEL)
REG = ["source", "repaired", "action-independent", "coupled"]

# Fig. 1 - logged controller.
beh = pd.read_csv("diag_behaviour.csv"); beh = beh[beh.band != "all"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.8, 2.6), dpi=220)
x = np.arange(len(beh))
a1.bar(x - .2, beh.dry_steps, .4, color="0.7", label="steps with a dry reading")
a1.bar(x + .2, beh.decision_steps, .4, color="C3", label="steps with a logged decision")
for i, p in enumerate(beh.p_decision_given_dry):
    a1.text(i, max(beh.dry_steps.iloc[i], beh.decision_steps.iloc[i]) + 3, f"{100 * p:.0f}%", ha="center", fontsize=6.5)
a1.set_xticks(x); a1.set_xticklabels(beh.band); a1.set_xlabel("step median soil moisture (%)"); a1.set_ylabel("10-min steps")
a1.legend(frameon=False, fontsize=6.5)
ctx = pd.read_csv("fault_context.csv")
a2.bar(np.arange(3) - .2, ctx.dry_reading_steps, .4, color="0.7", label="steps with a dry reading")
a2.bar(np.arange(3) + .2, ctx.decision_steps, .4, color="C3", label="steps with a logged decision")
for i, r in ctx.iterrows():
    a2.text(i + .2, r.decision_steps + 3, f"{100 * r.decision_steps / r.steps:.0f}% of\n{r.steps} steps", ha="center", fontsize=6)
a2.set_xticks(range(3)); a2.set_xticklabels(ctx.health); a2.set_xlabel("DHT22 health of the step"); a2.legend(frameon=False, fontsize=6.5)
fig.tight_layout(); fig.savefig(OUT / "fig1_behaviour.png", bbox_inches="tight"); plt.close(fig)

# Fig. 2 - fault chain: onset after a decision, onset after a dry reading without a decision, recovery.
on = pd.read_csv("fault_onset.csv").set_index("quantity")
cc = pd.read_csv("fault_commoncause.csv", index_col=0)["value"]
groups = [("failure onset,\nnext step", on.loc["onset1", "a0"], on.loc["onset1", "a1"], on.loc["onset1", "n_a0"], on.loc["onset1", "n_a1"], ("no decision", "decision")),
          ("failure onset,\nwithin 30 min", on.loc["onset3", "a0"], on.loc["onset3", "a1"], on.loc["onset3", "n_a0"], on.loc["onset3", "n_a1"], ("no decision", "decision")),
          ("onset within 30 min,\nno decision", cc["onset3_nodry"], cc["onset3_dry"], cc["n_nodry"], cc["n_dry"], ("no dry\nreading", "dry\nreading")),
          ("recovery,\nnext step", on.loc["recover1", "a0"], on.loc["recover1", "a1"], on.loc["recover1", "n_a0"], on.loc["recover1", "n_a1"], ("no decision", "decision"))]
fig, ax = plt.subplots(figsize=(6.2, 2.6), dpi=220)
for i, (lab, v0, v1, n0, n1, (l0, l1)) in enumerate(groups):
    c0, c1 = ("C0", "C3") if l1 == "decision" else ("0.6", "C1")
    ax.bar(i - .18, 100 * v0, .36, color=c0); ax.bar(i + .18, 100 * v1, .36, color=c1)
    ax.text(i - .18, 100 * v0 + .8, f"{l0}\nn={int(n0)}", ha="center", fontsize=5.2)
    ax.text(i + .18, 100 * v1 + .8, f"{l1}\nn={int(n1)}", ha="center", fontsize=5.2)
ax.set_xticks(range(len(groups))); ax.set_xticklabels([g[0] for g in groups], fontsize=7); ax.set_ylabel("probability (%)")
ax.set_ylim(0, 100 * on.loc["onset3", "a1"] + 8)
fig.tight_layout(); fig.savefig(OUT / "fig2_fault_chain.png", bbox_inches="tight"); plt.close(fig)

# Fig. 3 - tabular plan on the soil MDP (reference policy).
pol = pd.read_csv("mdp_policies.csv")
p0 = pol[pol["lambda"] == LAMBDAS[0]].sort_values("state")
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.6), dpi=220, gridspec_kw={"width_ratios": [1, 1.4]})
cov = p0.n_a1.to_numpy().reshape(len(BANDS), NF).T; cov0 = p0.n_a0.to_numpy().reshape(len(BANDS), NF).T
axes[0].imshow(cov, cmap="Reds", aspect="auto")
for i in range(NF):
    for j in range(len(BANDS)):
        axes[0].text(j, i, f"{cov[i, j]}/{cov0[i, j]}", ha="center", va="center", fontsize=6)
axes[0].set_xticks(range(len(BANDS))); axes[0].set_xticklabels(BANDS); axes[0].set_yticks(range(NF)); axes[0].set_yticklabels(FLEVELS)
axes[0].set_xlabel("moisture band (%)"); axes[0].set_title("logged transitions (a=1 / a=0)", fontsize=8)
mat = np.array([pol[pol["lambda"] == lam].sort_values("state").vi.to_numpy() for lam in LAMBDAS])
axes[1].imshow(mat, cmap="Greens", vmin=0, vmax=1.4, aspect="auto")
for i in range(len(LAMBDAS)):
    for j in range(mat.shape[1]):
        if mat[i, j]:
            axes[1].text(j, i, "I", ha="center", va="center", fontsize=7)
for b in range(1, len(BANDS)):
    axes[1].axvline(b * NF - .5, color="k", lw=.6)
axes[1].set_yticks(range(len(LAMBDAS))); axes[1].set_yticklabels([f"λ={l}" for l in LAMBDAS])
axes[1].set_xticks([b * NF + 1 for b in range(len(BANDS))]); axes[1].set_xticklabels(BANDS)
axes[1].set_xlabel("moisture band (%); within each band: f=0 | 0<f≤0.1 | f>0.1")
axes[1].set_title("optimal tabular policy: irrigate (I)", fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "fig3_policy.png", bbox_inches="tight"); plt.close(fig)

# Fig. 4 - decision rate on dry steps with and without a visible failure code (source regime).
ev = pd.read_csv("shortcut_eval.csv"); ev = ev[ev.seed >= 0]
src = ev[ev.regime == "source"].groupby("config")[["p_decide_dry_clean", "p_decide_dry_fault"]].agg(["mean", "std"]).reindex(ORDER)
fig, ax = plt.subplots(figsize=(6.6, 2.6), dpi=220)
x = np.arange(len(ORDER))
for k, (c, col, nm) in enumerate((("p_decide_dry_clean", "C0", "air sensor working"), ("p_decide_dry_fault", "C3", "failure code shown"))):
    mu, sd = 100 * src[(c, "mean")], 100 * src[(c, "std")]
    ax.bar(x + (k - .5) * .38, mu, .38, yerr=[np.minimum(mu, sd), np.minimum(100 - mu, sd)], capsize=1.5, color=col, label=nm, error_kw={"lw": .6})
ax.set_xticks(x); ax.set_xticklabels([LABEL[c] for c in ORDER], rotation=25, ha="right", fontsize=6.5)
ax.set_ylabel("P(decision | dry reading) (%)"); ax.legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "fig4_shortcut.png", bbox_inches="tight"); plt.close(fig)

# Fig. 5 - reward under the four sensor regimes: mean and 95% bootstrap interval over 15 seeds.
stt = pd.read_csv("shortcut_stats.csv").set_index(["config", "metric"])
fig, ax = plt.subplots(figsize=(6.8, 2.8), dpi=220)
for k, reg in enumerate(REG):
    m = np.array([stt.loc[(c, reg), "mean"] for c in ORDER]); lo = np.array([stt.loc[(c, reg), "mean_lo"] for c in ORDER])
    hi = np.array([stt.loc[(c, reg), "mean_hi"] for c in ORDER])
    ax.errorbar(x + (k - 1.5) * .17, 1000 * m, yerr=[1000 * (m - lo), 1000 * (hi - m)], fmt="o", ms=3, capsize=1.5, lw=.7, label=reg)
base = pd.read_csv("shortcut_eval.csv"); base = base[(base.seed < 0) & (base.regime == "source")].set_index("config")["reward"]
for nm, col in (("tabular plan", "C2"), ("any dry reading", "C1")):
    ax.axhline(1000 * base[nm], color=col, ls=":", lw=.8)
    ax.text(len(ORDER) - .45, 1000 * base[nm], nm, fontsize=5.5, va="center", color=col)
ax.set_xticks(x); ax.set_xticklabels([LABEL[c] for c in ORDER], rotation=25, ha="right", fontsize=6.5)
ax.set_ylabel("mean reward per step (× 10⁻³)"); ax.legend(frameon=False, fontsize=6.5, ncol=4, loc="lower left")
fig.tight_layout(); fig.savefig(OUT / "fig5_regimes.png", bbox_inches="tight"); plt.close(fig)

# Fig. 6 - counterfactual Q-gap (mean, 95% bootstrap interval) and learning curves.
RAW = [c for c in ORDER if c not in ("mask", "ffill")]
cu = pd.read_csv("shortcut_curve.csv")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.8, 2.6), dpi=220)
xr = np.arange(len(RAW))
m = np.array([stt.loc[(c, "mean_abs_dq"), "mean"] for c in RAW])
lo = np.array([stt.loc[(c, "mean_abs_dq"), "mean_lo"] for c in RAW]); hi = np.array([stt.loc[(c, "mean_abs_dq"), "mean_hi"] for c in RAW])
a1.bar(xr, m, .6, yerr=[m - lo, hi - m], capsize=1.5, color="C0", error_kw={"lw": .6})
a1.set_xticks(xr); a1.set_xticklabels([LABEL[c] for c in RAW], rotation=35, ha="right", fontsize=6)
a1.set_ylabel("CSI$_{ΔQ}$ (mean |ΔQ|)")
for (cfg, col) in (("full", "C3"), ("sc", "C2")):
    z = cu[cu.config == cfg].groupby("iteration").reward
    a2.plot(z.mean().index, 1000 * z.mean(), color=col, lw=1.2, label=LABEL[cfg])
    a2.fill_between(z.mean().index, 1000 * z.quantile(.25), 1000 * z.quantile(.75), color=col, alpha=.2)
a2.set_xlabel("training iteration"); a2.set_ylabel("reward per step, source (× 10⁻³)"); a2.legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "fig6_csi_curves.png", bbox_inches="tight"); plt.close(fig)
print("figures written to", OUT.resolve())


In [ ]:
# Helper: run a script from work/src inside work/outputs and stream its output.
def run(script, *args):
    cmd = [sys.executable, "-u", str(SRC / script), *map(str, args)]
    env = dict(os.environ, MPLBACKEND="Agg", PYTHONIOENCODING="utf-8")
    p = subprocess.Popen(cmd, cwd=OUT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed")


## 4. Run the experiment
All outputs go to `work/outputs/`.

### Step 1 - cleaning and 10-min decision steps (Table 2)

In [ ]:
run("rl_prep.py", DATA)

### Step 2 - diagnostics: controller, Markov order, air readings, decision effect (Table 3, Fig. 1)

In [ ]:
run("diagnostics.py")

### Step 3 - action-dependent Markov chain of DHT22 health, common-cause check (Table 4, Fig. 2)

In [ ]:
run("fault_chain.py")

### Step 4 - tabular soil MDP: reference plan and day bootstrap (Fig. 3)

In [ ]:
run("mdp_planning.py")

### Step 5 - simulator audit (Tables 5-6)

In [ ]:
run("sim_audit.py")

### Step 6 - DQN variants, 15 seeds, four sensor regimes, shortcut index (Tables 7-8, Figs. 4-6)

In [ ]:
run("drl_shortcut.py")

### Step 7 - robustness check in the band-constrained simulator (Sec. 7.4)

In [ ]:
RUN_ROBUSTNESS = True
if RUN_ROBUSTNESS:
    run("drl_robustness.py")
else:
    print("Skipped.")

### Step 8 - figures

In [ ]:
run("figures_rl.py")

## 5. Check against the revised paper

In [ ]:
# Compare every reported number with what this run produced.
import math
import pandas as pd
CHECKS = []

def chk(item, expected, obtained, tol=0.006):
    """Numbers are compared at the precision printed in the paper."""
    if isinstance(expected, str):
        ok = str(obtained) == expected
    else:
        ok = obtained is not None and not (isinstance(obtained, float) and math.isnan(obtained)) and abs(obtained - expected) <= tol
    CHECKS.append({"item": item, "paper": expected,
                   "this run": round(obtained, 4) if isinstance(obtained, float) else obtained,
                   "status": "✅" if ok else "❌"})

PS = pd.read_csv(OUT / "prep_summary.csv", index_col=0)["value"]
for k, v in {"clean_readings": 53189, "decisions": 208, "steps": 784, "decision_steps": 71, "transitions": 753,
             "decision_transitions": 68, "days": 16, "steps_with_dry_reading": 260, "decision_steps_with_median_ge_50": 70,
             "steps_healthy": 667, "steps_degraded": 17, "steps_failed": 100}.items():
    chk(f"Table 2 | {k}", v, int(PS[k]), 0)
CTX = pd.read_csv(OUT / "fault_context.csv").set_index("health")
for h, v in {"healthy": 16, "degraded": 9, "failed": 46}.items():
    chk(f"Table 2 | decision steps, {h}", v, int(CTX.loc[h, "decision_steps"]), 0)
BEH = pd.read_csv(OUT / "diag_behaviour.csv").set_index("band")
for b, v in {"<60": 0.083, "70-80": 0.581, "80-90": 0.645}.items():
    chk(f"Sec. 3.2 | P(decision | dry), {b}", v, BEH.loc[b, "p_decision_given_dry"], 0.0006)

MK = pd.read_csv(OUT / "diag_markov.csv").set_index(["target", "features", "model"])
T3 = {"M0: m, f, a": (3.4312, 3.3402, 0.0348, 0.0369), "M1: + previous decision": (3.4293, 3.3676, 0.0348, 0.0357),
      "M2: + previous m and f": (3.4226, 3.3438, 0.0341, 0.0363), "M3: + hour of day": (3.4706, 3.6651, 0.0351, 0.0403),
      "A0: m, f, a": (3.1135, 3.1317, 0.0312, 0.0349), "A1: + humidity": (3.1540, 3.2960, 0.0328, 0.0318),
      "A2: + humidity, air temperature": (3.1590, 3.5031, 0.0332, 0.0325)}
for fs, vals in T3.items():
    for (tg, mdl), v in zip([("d1", "linear"), ("d1", "boosting"), ("f1", "linear"), ("f1", "boosting")], vals):
        chk(f"Table 3 | {fs} | {tg} {mdl}", v, MK.loc[(tg, fs, mdl), "MAE"], 0.00006)
chk("Table 3 | transitions with a healthy sensor", 639, int(MK.loc[("f1", "A0: m, f, a", "linear"), "n"]), 0)
EF = pd.read_csv(OUT / "diag_effect.csv").set_index("outcome")
chk("Sec. 3.3 | within-state effect on f1", -0.0097, EF.loc["f1", "within_state"], 0.00006)
chk("Sec. 3.3 | effect interval low", -0.0835, EF.loc["f1", "lo"], 0.00006)
chk("Sec. 3.3 | effect interval high", 0.0193, EF.loc["f1", "hi"], 0.00006)
INS = pd.read_csv(OUT / "mdp_insample.csv"); INS = INS[INS["lambda"] == 0.05].set_index("policy")
for p, v in {"value iteration": -0.0340, "logged controller": -0.0492, "never": -0.0515}.items():
    chk(f"Sec. 3.3 | tabular J | {p}", v, INS.loc[p, "J"], 0.00006)
BO = pd.read_csv(OUT / "mdp_bootstrap.csv"); BO = BO[BO["lambda"] == 0.05].set_index("comparison")
chk("Sec. 3.3 | P(plan beats controller)", 0.862, BO.loc["VI - logged controller", "p_positive"], 0.0006)

FC = pd.read_csv(OUT / "fault_chain.csv").set_index(["action", "from", "to"])
T4 = {(0, "healthy"): ((0.976, 610), (0.017, 10), (0.007, 4)), (0, "degraded"): ((0.684, 6), (0.158, 1), (0.158, 1)),
      (0, "failed"): ((0.046, 2), (0.028, 1), (0.927, 50)), (1, "healthy"): ((0.879, 14), (0.091, 1), (0.030, 0)),
      (1, "degraded"): ((0.333, 3), (0.238, 2), (0.429, 4)), (1, "failed"): ((0.077, 3), (0.011, 0), (0.912, 41))}
for (a, fr), cells in T4.items():
    for to, (p, n) in zip(("healthy", "degraded", "failed"), cells):
        chk(f"Table 4 | a={a} {fr}->{to} | p", p, FC.loc[(a, fr, to), "p"], 0.0006)
        chk(f"Table 4 | a={a} {fr}->{to} | count", n, int(FC.loc[(a, fr, to), "count"]), 0)
ON = pd.read_csv(OUT / "fault_onset.csv").set_index("quantity")
for q, (a0, a1, n0, n1) in {"onset1": (0.0224, 0.0667, 624, 15), "onset3": (0.0547, 0.2143, 585, 14), "recover1": (0.0377, 0.0682, 53, 44)}.items():
    chk(f"Sec. 3.4 | {q} | no decision", a0, ON.loc[q, "a0"], 0.00006); chk(f"Sec. 3.4 | {q} | decision", a1, ON.loc[q, "a1"], 0.00006)
    chk(f"Sec. 3.4 | {q} | n no decision", n0, int(ON.loc[q, "n_a0"]), 0); chk(f"Sec. 3.4 | {q} | n decision", n1, int(ON.loc[q, "n_a1"]), 0)
chk("Sec. 3.4 | 30-min onset difference, low", -0.052, ON.loc["onset3", "diff_lo"], 0.0006)
chk("Sec. 3.4 | 30-min onset difference, high", 0.820, ON.loc["onset3", "diff_hi"], 0.0006)
chk("Sec. 3.4 | 30-min onset difference, P(>0)", 0.873, ON.loc["onset3", "p_diff_positive"], 0.0006)
CC = pd.read_csv(OUT / "fault_commoncause.csv", index_col=0)["value"]
for k, v, tol in (("onset3_dry", 0.0549, 0.00006), ("onset3_nodry", 0.0546, 0.00006), ("n_dry", 164, 0), ("n_nodry", 421, 0),
                  ("diff3_lo", -0.103, 0.0006), ("diff3_hi", 0.080, 0.0006)):
    chk(f"Sec. 3.5 | common cause | {k}", v, CC[k], tol)
chk("Sec. 3.5 | common cause | difference (points)", 0.02, 100 * CC["diff3"], 0.006)

AE = pd.read_csv(OUT / "audit_effect.csv").set_index("band")
T7 = {"effect: replay k=10": (-0.123, 0.003, -0.011, 0.023, 0.012, -0.032), "effect: replay k=5": (-0.108, -0.003, -0.015, 0.020, -0.002, -0.031),
      "effect: replay k=30": (-0.171, 0.004, 0.014, 0.035, 0.032, -0.038), "effect: band-constrained k=10": (-0.065, 0.040, -0.006, 0.025, -0.004, -0.000),
      "effect: tabular MDP": (-0.077, 0.000, -0.007, 0.010, 0.000, -0.021), "mean distance to a=1 neighbours": (0.78, 0.39, 0.23, 0.14, 0.37, 0.44),
      "mean distance to a=0 neighbours": (0.20, 0.09, 0.22, 0.14, 0.10, 0.15), "share of a=1 neighbours from another band": (0.53, 0.36, 0.12, 0.14, 0.48, 0.35)}
BANDS7 = ["<60", "60-70", "70-80", "80-90", ">=90", "all"]
for col, vals in T7.items():
    tol = 0.0006 if col.startswith("effect") else 0.006
    for b, v in zip(BANDS7, vals):
        chk(f"Table 5 | {col} | {b}", v, AE.loc[b, col], tol)
chk("Table 5 | dry steps", 254, int(AE.loc["all", "dry_steps"]), 0)
chk("Sec. 5 | effect, band-constrained (4 decimals)", -0.0003, AE.loc["all", "effect: band-constrained k=10"], 0.00006)
chk("Sec. 5 | decision neighbours >10 points wetter, <60", 0.19, AE.loc["<60", "share of a=1 neighbours >10 points wetter"], 0.006)
AP = pd.read_csv(OUT / "audit_policies.csv").set_index(["model", "policy"])
POL = ["never", "logged controller (clone)", "any dry reading", "tabular plan"]
T8 = {"tabular MDP, discounted J": ((0.0515, 0.0448, 0.0296, 0.0287), (0, 12.6, 41.1, 15.5)),
      "tabular MDP, 144-step days": ((0.0506, 0.0426, 0.0255, 0.0237), (0, 12.6, 37.1, 12.8)),
      "replay k=10": ((0.0634, 0.0462, 0.0208, 0.0210), (0, 15.1, 41.9, 11.6)),
      "replay k=5": ((0.0185, 0.0138, 0.0052, 0.0045), (0, 4.4, 10.1, 2.6)),
      "replay k=30": ((0.0561, 0.0475, 0.0261, 0.0240), (0, 15.2, 49.0, 15.1)),
      "band-constrained k=10": ((0.0591, 0.0506, 0.0366, 0.0322), (0, 14.8, 50.1, 16.1)),
      "null effect k=10": ((0.0634, 0.0591, 0.0609, 0.0604), (0, 13.6, 53.4, 25.6))}
for mdl, (dry, dec) in T8.items():
    for p, v in zip(POL, dry):
        chk(f"Table 6 | {mdl} | {p} | dry share", v, AP.loc[(mdl, p), "dry_share"], 0.00006)
    for p, v in zip(POL, dec):
        chk(f"Table 6 | {mdl} | {p} | decisions/day", v, AP.loc[(mdl, p), "decisions_per_day"], 0.06)
for mdl, v in {"tabular MDP, 144-step days": 0.0269, "replay k=10": 0.0425, "band-constrained k=10": 0.0269, "null effect k=10": 0.0030}.items():
    chk(f"Sec. 5 | never - plan gap | {mdl}", v, AP.loc[(mdl, "never"), "dry_share"] - AP.loc[(mdl, "tabular plan"), "dry_share"], 0.00006)
chk("Sec. 5 | null effect, plan reward", -0.0693, AP.loc[("null effect k=10", "tabular plan"), "reward"], 0.00006)

ST = pd.read_csv(OUT / "shortcut_stats.csv").set_index(["config", "metric"])
T10 = {"full": (-26.5, -24.9, -26.2, -29.3, -29.5, -27.6, -32.9, -26.8, 4.8, 0.117, 35.4),
       "mask": (-26.4, -26.4, -26.4, -26.4, -26.4, -23.7, -31.4, -23.2, 0.0, None, None),
       "ffill": (-25.1, -24.9, -24.9, -25.6, -26.0, -25.7, -27.1, -25.1, 1.7, None, None),
       "sc": (-25.6, -26.1, -25.6, -26.0, -26.8, -26.0, -28.4, -25.4, 1.8, 0.015, 11.7),
       "sc+RAE": (-26.1, -26.2, -25.9, -26.0, -27.1, -26.2, -28.8, -25.6, 2.0, 0.015, 12.7),
       "full+RAE": (-27.0, -25.4, -26.6, -29.5, -29.8, -28.9, -31.9, -28.0, 4.8, 0.119, 43.3),
       "sc-noAD": (-26.2, -25.5, -26.1, -25.9, -27.0, -26.3, -28.2, -25.9, 2.0, 0.016, 15.4),
       "sc-b0.1": (-25.3, -24.7, -25.5, -25.8, -26.6, -26.6, -27.9, -25.5, 2.4, 0.042, 25.9),
       "sc-b10": (-31.9, -31.1, -31.5, -31.1, -32.4, -31.7, -34.8, -30.5, 2.2, 0.013, 7.4)}
keys = [("source", "mean", 1000), ("repaired", "mean", 1000), ("action-independent", "mean", 1000), ("coupled", "mean", 1000),
        ("worst", "mean", 1000), ("worst", "iqm", 1000), ("worst", "mean_lo", 1000), ("worst", "mean_hi", 1000),
        ("spread", "mean", 1000), ("mean_abs_dq", "mean", 1), ("flip_rate", "mean", 100)]
for cfg, vals in T10.items():
    for (metric, stat, k), v in zip(keys, vals):
        if v is not None:
            chk(f"Table 8 | {cfg} | {metric} {stat}", v, k * ST.loc[(cfg, metric), stat], 0.06 if k != 1 else 0.0006)
for cfg, metric, stat, k, v in (("full", "mean_abs_dq", "mean_lo", 1, 0.088), ("full", "mean_abs_dq", "mean_hi", 1, 0.149),
                                ("full", "spread", "mean_lo", 1000, 2.8), ("full", "spread", "mean_hi", 1000, 7.3),
                                ("sc", "mean_abs_dq", "mean_lo", 1, 0.014), ("sc", "mean_abs_dq", "mean_hi", 1, 0.017)):
    chk(f"Sec. 7 | {cfg} | {metric} {stat}", v, k * ST.loc[(cfg, metric), stat], 0.06 if k != 1 else 0.0006)
EB = pd.read_csv(OUT / "shortcut_eval.csv")
FX = EB[(EB.seed < 0) & (EB.regime == "source")].set_index("config")
for c, v in {"tabular plan": -25.0, "any dry reading": -35.4, "logged controller (clone)": -51.4, "never": -63.4}.items():
    chk(f"Table 8 | fixed | {c}", v, 1000 * FX.loc[c, "reward"], 0.06)
for c, v in {"tabular plan": 0.246, "any dry reading": 0.088, "logged controller (clone)": 0.572, "never": 0.848}.items():
    chk(f"Sec. 7.5 | shortfall | {c}", v, FX.loc[c, "deficit"], 0.0006)
chk("Sec. 7.5 | any dry reading, decisions per day", 41.9, FX.loc["any dry reading", "decisions_per_day"], 0.06)
EV = EB[EB.seed >= 0]
SRC = EV[EV.regime == "source"].groupby("config")[["p_decide_dry_clean", "p_decide_dry_fault", "deficit"]].agg(["mean", "std"])
chk("Sec. 7.1 | DQN P(decide | dry, clean)", 38.6, 100 * SRC.loc["full", ("p_decide_dry_clean", "mean")], 0.06)
chk("Sec. 7.1 | DQN P(decide | dry, fault)", 52.8, 100 * SRC.loc["full", ("p_decide_dry_fault", "mean")], 0.06)
chk("Sec. 7.1 | DQN P(decide | dry, fault) s.d.", 36.8, 100 * SRC.loc["full", ("p_decide_dry_fault", "std")], 0.06)
for c, v in {"full": 0.189, "sc": 0.188, "ffill": 0.161, "mask": 0.199}.items():
    chk(f"Sec. 7.5 | shortfall | {c}", v, SRC.loc[c, ("deficit", "mean")], 0.0006)
PSD = pd.read_csv(OUT / "shortcut_per_seed.csv")
chk("Sec. 7.1 | DQN seeds with spread > 5", 5, int((PSD[PSD.config == "full"].spread > 0.005).sum()), 0)
chk("Sec. 7.2 | SC-DQN seeds with spread > 5", 0, int((PSD[PSD.config == "sc"].spread > 0.005).sum()), 0)
chk("Sec. 7.2 | masking, failed seed", -55.9, 1000 * PSD[PSD.config == "mask"].source.min(), 0.06)
TS = pd.read_csv(OUT / "shortcut_tests.csv").set_index(["a", "b", "metric"])
for key, v, tol in ((("sc", "full", "spread"), 0.041, 0.0006), (("sc", "full", "worst"), 0.068, 0.0006),
                    (("sc", "full", "coupled"), 0.021, 0.0006), (("full+RAE", "full", "flip_rate"), 0.079, 0.0006),
                    (("ffill", "sc", "worst"), 0.68, 0.006), (("sc-b0.1", "sc", "source"), 0.90, 0.006)):
    chk(f"Sec. 7 | Mann-Whitney p | {' vs '.join(key[:2])} {key[2]}", v, TS.loc[key, "p"], tol)
chk("Sec. 7.2 | Mann-Whitney p | sc vs full CSI < 0.001", "True", str(bool(TS.loc[("sc", "full", "mean_abs_dq"), "p"] < 0.001)))
CU = pd.read_csv(OUT / "shortcut_curve.csv").groupby(["config", "iteration"]).reward.mean() * 1000
for (cfg, it), v in {("full", 500): -35.0, ("full", 3000): -26.6, ("sc", 500): -38.3, ("sc", 3000): -25.8}.items():
    chk(f"Sec. 7.5 | {cfg} curve, iteration {it}", v, CU.loc[(cfg, it)], 0.06)
RS = pd.read_csv(OUT / "robust_stats.csv").set_index(["config", "metric"])
T9 = {"full": (-40.5, -37.9, -39.1, -43.4, -44.1, -46.4, -41.9, 6.8, 0.152, 32.6),
      "mask": (-35.8, -35.8, -35.8, -35.8, -35.8, -38.7, -34.0, 0.0, None, None),
      "ffill": (-37.8, -38.0, -38.0, -37.6, -39.0, -41.2, -36.9, 2.3, None, None),
      "sc": (-38.3, -38.0, -38.9, -38.7, -40.3, -41.8, -38.6, 3.4, 0.019, 15.7)}
rkeys = [("source", "mean", 1000), ("repaired", "mean", 1000), ("action-independent", "mean", 1000), ("coupled", "mean", 1000),
         ("worst", "mean", 1000), ("worst", "mean_lo", 1000), ("worst", "mean_hi", 1000), ("spread", "mean", 1000),
         ("mean_abs_dq", "mean", 1), ("flip_rate", "mean", 100)]
for cfg, vals in T9.items():
    for (metric, stat, k), v in zip(rkeys, vals):
        if v is not None:
            chk(f"Table 9 | {cfg} | {metric} {stat}", v, k * RS.loc[(cfg, metric), stat], 0.06 if k != 1 else 0.0006)
chk("Sec. 7.4 | DQN spread interval low", 5.7, 1000 * RS.loc[("full", "spread"), "mean_lo"], 0.06)
chk("Sec. 7.4 | DQN spread interval high", 8.1, 1000 * RS.loc[("full", "spread"), "mean_hi"], 0.06)
RE = pd.read_csv(OUT / "robust_eval.csv"); RF = RE[(RE.seed < 0) & (RE.regime == "source")].set_index("config")
chk("Table 9 | tabular plan", -37.8, 1000 * RF.loc["tabular plan", "reward"], 0.06)
chk("Table 9 | never", -59.1, 1000 * RF.loc["never", "reward"], 0.06)
RP = pd.read_csv(OUT / "robust_per_seed.csv")
chk("Sec. 7.4 | DQN seeds with spread > 5", 8, int((RP[RP.config == "full"].spread > 0.005).sum()), 0)
chk("Sec. 7.4 | SC-DQN seeds with spread > 5", 2, int((RP[RP.config == "sc"].spread > 0.005).sum()), 0)
chk("Sec. 7.4 | seeds per agent", 10, int((RP.config == "full").sum()), 0)
RT = pd.read_csv(OUT / "robust_tests.csv").set_index("metric")
for m, v, tol in (("mean_abs_dq", 0.0001, 0.00006), ("spread", 0.0007, 0.00006), ("worst", 0.045, 0.0006)):
    chk(f"Sec. 7.4 | Mann-Whitney p | sc vs full {m}", v, RT.loc[m, "p"], tol)


In [ ]:
report = pd.DataFrame(CHECKS)
n_ok = (report.status == "✅").sum()
print(f"{n_ok} / {len(report)} values match the revised paper.")
pd.set_option("display.max_rows", 500)
report[report.status == "❌"] if n_ok < len(report) else print("All checks passed.")


Full list of checks:

In [ ]:
report

## 6. Figures

In [ ]:
# Figures produced by this run
from IPython.display import Image, display
for f in sorted(OUT.rglob("*.png")):
    print(f.name); display(Image(filename=str(f), width=520))
